# **Task 1: Tải và Tiền xử lý Dữ liệu**

In [9]:
# 1. Hàm đọc dữ liệu CoNLL2003 (format IOB2)
def load_conll2003(path):
    sentences = []
    tags = []

    with open(path, "r", encoding="utf-8") as f:
        words, ner_tags = [], []

        for line in f:
            line = line.strip()

            # Dòng rỗng => kết thúc câu
            if not line:
                if words:
                    sentences.append(words)
                    tags.append(ner_tags)
                    words, ner_tags = [], []
                continue

            # Format CoNLL: word POS CHUNK NER
            parts = line.split()
            word = parts[0]
            tag = parts[-1]  # NER tag nằm cuối

            words.append(word)
            ner_tags.append(tag)

    return sentences, tags

In [12]:
# 2. Load train / valid / test
train_path = "../data/conll2003/train.txt"
valid_path = "../data/conll2003/valid.txt"
test_path  = "../data/conll2003/test.txt"

train_sentences, train_tags = load_conll2003(train_path)
valid_sentences, valid_tags = load_conll2003(valid_path)
test_sentences,  test_tags  = load_conll2003(test_path)

print("Train sentences:", len(train_sentences))
print("Valid sentences:", len(valid_sentences))
print("Test sentences:", len(test_sentences))


Train sentences: 14987
Valid sentences: 3466
Test sentences: 3684


In [13]:
# 3. Xây dựng vocabulary
word_to_ix = {"<PAD>": 0, "<UNK>": 1}
tag_to_ix  = {"<PAD>": 0}

# Words
for sent in train_sentences:
    for w in sent:
        if w not in word_to_ix:
            word_to_ix[w] = len(word_to_ix)

# Tags
for seq in train_tags:
    for t in seq:
        if t not in tag_to_ix:
            tag_to_ix[t] = len(tag_to_ix)

print("Word vocab size:", len(word_to_ix))
print("Tag vocab size:", len(tag_to_ix))


Word vocab size: 23626
Tag vocab size: 10


# **Task 2: Tạo PyTorch Dataset và DataLoader**

In [14]:
# 2.1 Dataset class
import torch
from torch.utils.data import Dataset

class NERDataset(Dataset):
    def __init__(self, sentences, tags, word_to_ix, tag_to_ix):
        self.sentences = sentences
        self.tags = tags
        self.word_to_ix = word_to_ix
        self.tag_to_ix = tag_to_ix

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        words = self.sentences[idx]
        tags  = self.tags[idx]

        word_ids = [self.word_to_ix.get(w, self.word_to_ix["<UNK>"]) for w in words]
        tag_ids  = [self.tag_to_ix[t] for t in tags]

        return torch.tensor(word_ids), torch.tensor(tag_ids)


In [15]:
# 2.2 collate_fn (padding)
from torch.nn.utils.rnn import pad_sequence

PAD_WORD = word_to_ix["<PAD>"]
PAD_TAG  = tag_to_ix["<PAD>"]

def collate_fn(batch):
    sents, tags = zip(*batch)

    sent_pad = pad_sequence(sents, batch_first=True, padding_value=PAD_WORD)
    tag_pad  = pad_sequence(tags,  batch_first=True, padding_value=PAD_TAG)

    lengths = torch.tensor([len(s) for s in sents])

    return sent_pad, tag_pad, lengths


In [16]:
# 2.3 DataLoader
from torch.utils.data import DataLoader

train_dataset = NERDataset(train_sentences, train_tags, word_to_ix, tag_to_ix)
valid_dataset = NERDataset(valid_sentences, valid_tags, word_to_ix, tag_to_ix)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  collate_fn=collate_fn)
valid_loader = DataLoader(valid_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)


# **Task 3: Xây dựng Mô hình RNN**

In [17]:
import torch.nn as nn

class SimpleRNNForNER(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim, num_tags):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.rnn = nn.RNN(emb_dim, hidden_dim, batch_first=True)
        self.fc  = nn.Linear(hidden_dim, num_tags)

    def forward(self, x, lengths):
        emb = self.embedding(x)

        packed = nn.utils.rnn.pack_padded_sequence(
            emb, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        out, _ = self.rnn(packed)

        out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True)
        logits = self.fc(out)

        return logits


# **Task 4: Huấn luyện Mô hình**

In [18]:
# 4.1 Khởi tạo mô hình + loss + optimizer
device = "cuda" if torch.cuda.is_available() else "cpu"

model = SimpleRNNForNER(
    vocab_size=len(word_to_ix),
    emb_dim=128,
    hidden_dim=128,
    num_tags=len(tag_to_ix)
).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=PAD_TAG)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


In [21]:
# 4.2 Training loop
for epoch in range(3):
    model.train()
    total_loss = 0

    for words, tags, lengths in train_loader:
        words, tags = words.to(device), tags.to(device)

        optimizer.zero_grad()

        outputs = model(words, lengths)
        outputs = outputs.view(-1, outputs.shape[-1])
        tags = tags.view(-1)

        loss = criterion(outputs, tags)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} - Loss: {total_loss/len(train_loader):.4f}")


Epoch 1 - Loss: 0.1976
Epoch 2 - Loss: 0.1391
Epoch 3 - Loss: 0.1000


# **Task 5: Đánh giá Mô hình**

In [27]:
# !pip install seqeval

In [28]:
# 5.1 Hàm evaluate
from seqeval.metrics import classification_report, precision_score, recall_score, f1_score

def evaluate_full(model, loader, ix_to_tag):
    model.eval()

    all_true = []
    all_pred = []

    total_correct = 0
    total_tokens = 0

    with torch.no_grad():
        for words, tags, lengths in loader:
            words, tags = words.to(device), tags.to(device)

            outputs = model(words, lengths)
            preds = outputs.argmax(-1)

            for i in range(words.size(0)):  
                length = lengths[i].item()

                true_seq = tags[i][:length].tolist()
                pred_seq = preds[i][:length].tolist()

                true_labels = [ix_to_tag[t] for t in true_seq]
                pred_labels = [ix_to_tag[p] for p in pred_seq]

                all_true.append(true_labels)
                all_pred.append(pred_labels)

                total_correct += sum([1 for t, p in zip(true_seq, pred_seq) if t == p])
                total_tokens += length

    accuracy = total_correct / total_tokens

    print("=== REPORT THEO THỰC THỂ (seqeval) ===\n")
    print(classification_report(all_true, all_pred))

    precision = precision_score(all_true, all_pred)
    recall = recall_score(all_true, all_pred)
    f1 = f1_score(all_true, all_pred)

    print("\n=== TỔNG KẾT ===")
    print("Accuracy:", accuracy)
    print("Precision:", precision)
    print("Recall:", recall)
    print("F1-score:", f1)

    return accuracy, precision, recall, f1


In [29]:
ix_to_tag = {v: k for k, v in tag_to_ix.items()}

acc, pre, rec, f1 = evaluate_full(model, valid_loader, ix_to_tag)


=== REPORT THEO THỰC THỂ (seqeval) ===

              precision    recall  f1-score   support

         LOC       0.42      0.80      0.55      1837
        MISC       0.50      0.63      0.56       922
         ORG       0.47      0.58      0.52      1341
         PER       0.73      0.60      0.66      1842

   micro avg       0.50      0.66      0.57      5942
   macro avg       0.53      0.65      0.57      5942
weighted avg       0.54      0.66      0.58      5942


=== TỔNG KẾT ===
Accuracy: 0.912986156888596
Precision: 0.5049441376653396
Recall: 0.6617300572197913
F1-score: 0.5728020977492898


In [30]:
# 5.2 Dự đoán câu mới
def predict_sentence(sentence):
    tokens = sentence.split()
    ids = [word_to_ix.get(w, word_to_ix["<UNK>"]) for w in tokens]

    ids = torch.tensor(ids).unsqueeze(0).to(device)
    lengths = torch.tensor([len(tokens)])

    with torch.no_grad():
        outputs = model(ids, lengths)
    
    preds = outputs.argmax(-1).squeeze(0)

    ix_to_tag = {v: k for k, v in tag_to_ix.items()}
    pred_tags = [ix_to_tag[p.item()] for p in preds]

    return list(zip(tokens, pred_tags))

predict_sentence("VNU University is located in Hanoi")

[('VNU', 'B-ORG'),
 ('University', 'I-ORG'),
 ('is', 'O'),
 ('located', 'O'),
 ('in', 'O'),
 ('Hanoi', 'B-LOC')]